In [1]:
import pandas as pd
import numpy as np

In [2]:
linarg_stats = pd.read_csv('../data/ukb20279_linarg_stats.tsv', sep='\t')
disk_sizes = pd.read_csv('../data/ukb20279_vcf_pgen_sizes.tsv', sep='\t')

def get_row(name, n_samples, linarg_name, chrom_filter=None, dataset='all'):
    if chrom_filter:
        ls = linarg_stats[(linarg_stats.linarg == linarg_name) & (linarg_stats.chrom.isin(chrom_filter))]
    else:
        ls = linarg_stats[linarg_stats.linarg == linarg_name]
    
    ds = disk_sizes[disk_sizes.dataset == dataset]
    if chrom_filter:
        ds = ds[ds.chrom.isin(chrom_filter)]
    
    geno_nnz = np.sum(ls.genotypes_nnz)
    la_nnz = np.sum(ls.linarg_nnz)
    
    return {
        'Dataset': name,
        'Number of samples': f'{n_samples:,}',
        'Number of variants': f'{int(np.sum(ls.n_variants)):,}',
        'NNZ ratio': round(geno_nnz / la_nnz, 1),
        'Linear ARG size in memory (GB)': round(np.sum(ls.size_in_memory), 1),
        'Linear ARG size on disk (GB)': round(np.sum(ls.linarg_disk_size), 1),
        'Linear ARG metadata size on disk (GB)': round(np.sum(ls.variant_disk_size), 1),
        'vcf.gz size on disk (GB)': round(np.sum(ds.vcf_size), 1),
    }

rows = [
    get_row('UK Biobank 200k', 199911, 'ukb20279_chr1-22'),
    get_row('UK Biobank 200k MAF > 0.01', 199911, 'ukb20279_maf_0.01_chr1-22', dataset='common'),
    get_row('UK Biobank 200k chr 1, 11, 21', 199911, 'ukb20279_chr1-22', chrom_filter=[1, 11, 21]),
    get_row('UK Biobank 200k chr 21', 199911, 'ukb20279_chr1-22', chrom_filter=[21]),
]

table = pd.DataFrame(rows)
table

,Dataset,Number of samples,Number of variants,NNZ ratio,Linear ARG size in memory (GB),Linear ARG size on disk (GB),Linear ARG metadata size on disk (GB),vcf.gz size on disk (GB)
0,UK Biobank 200k,"199,911","659,887,507",75.6,58.0,26.5,3.5,2347.1
1,UK Biobank 200k MAF > 0.01,"199,911","9,886,035",130.7,30.1,12.9,0.1,527.6
2,"UK Biobank 200k chr 1, 11, 21","199,911","93,742,553",75.5,8.2,3.7,0.5,333.7
3,UK Biobank 200k chr 21,"199,911","8,463,891",70.0,0.9,0.4,0.0,30.7


In [3]:
# 1000 Genomes row - computed from h5 file
import linear_dag as ld
from linear_dag.core.lineararg import list_blocks
import h5py, os

linarg_path = '/Users/ambershen/Desktop/linARG/data/1kg/1kg_linear_arg.h5'

linarg_nnz = 0
genotypes_nnz = 0
n_variants = 0
block_metadata = list_blocks(linarg_path)

for block in block_metadata.get_column("block_name").to_list():
    with h5py.File(linarg_path, 'r') as f:
        linarg_nnz += f[block].attrs["n_entries"]
        n_variants += f[block].attrs["n_variants"]
    linarg = ld.LinearARG.read(linarg_path, block=block)
    linarg.flip = np.zeros(linarg.flip.shape, dtype=bool)
    v = np.ones(linarg.shape[0])
    allele_counts = v @ linarg
    genotypes_nnz += np.sum(allele_counts)

# Get disk size of h5 file
h5_size_gb = os.path.getsize(linarg_path) / 1e9

nnz_ratio_1kg = genotypes_nnz / linarg_nnz
print(f'1KG: {n_variants:,} variants, NNZ ratio: {nnz_ratio_1kg:.1f}')

1KG: 70,692,015 variants, NNZ ratio: 15.9


In [ ]:
# Build final table matching manuscript Table 1
row_1kg = {
    'Dataset': '1000 Genomes',
    'Number of samples': '6,404',
    'Number of variants': f'{n_variants:,}',
    'NNZ ratio': round(nnz_ratio_1kg, 1),
    'Linear ARG size in memory (GB)': 5.7,
    'Linear ARG size on disk (GB)': 2.1,
    'Linear ARG metadata size on disk (GB)': 0.5,
    'vcf.gz size on disk (GB)': 35.0,
}

row_500k = {
    'Dataset': 'UK Biobank 500k chr 21',
    'Number of samples': '490,541',
    'Number of variants': '9,479,292',
    'NNZ ratio': 72.7,
    'Linear ARG size in memory (GB)': 2.16,
    'Linear ARG size on disk (GB)': 1.0,
    'Linear ARG metadata size on disk (GB)': 0.1,
    'vcf.gz size on disk (GB)': 49.5,
}

final_table = pd.DataFrame([row_1kg] + rows + [row_500k])
final_table

,Dataset,Number of samples,Number of variants,NNZ ratio,Linear ARG size in memory (GB),Linear ARG size on disk (GB),Linear ARG metadata size on disk (GB),vcf.gz size on disk (GB)
0,1000 Genomes,"6,404","70,692,015",15.9,5.70,2.1,0.5,35.0
1,UK Biobank 200k,"199,911","659,887,507",75.6,58.00,26.5,3.5,2347.1
2,UK Biobank 200k MAF > 0.01,"199,911","9,886,035",130.7,30.10,12.9,0.1,527.6
3,"UK Biobank 200k chr 1, 11, 21","199,911","93,742,553",75.5,8.20,3.7,0.5,333.7
4,UK Biobank 200k chr 21,"199,911","8,463,891",70.0,0.90,0.4,0.0,30.7
5,UK Biobank 500k chr 21,"490,541","9,479,292",72.7,2.16,1.0,0.1,49.5
